In [1]:
import pandas as pd
import numpy as np

processed_path = "C:/Users/user/mutual-fund-analytics/data/processed/"

nav = pd.read_csv(processed_path + "02_nav_history_cleaned.csv")
nav['date'] = pd.to_datetime(nav['date'])
nav = nav.sort_values(['amfi_code', 'date'])

# Calculate daily returns: (NAV_today / NAV_yesterday) - 1
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()

print(nav.shape)
print(nav['daily_return'].describe())

(64320, 4)
count    64280.000000
mean         0.000451
std          0.008706
min         -0.058102
25%         -0.002092
50%          0.000000
75%          0.003233
max          0.064713
Name: daily_return, dtype: float64


In [2]:
# Calculate CAGR for 1yr, 3yr, 5yr per fund
results = []

for code in nav['amfi_code'].unique():
    fund = nav[nav['amfi_code'] == code].sort_values('date')
    end_nav = fund['nav'].iloc[-1]
    end_date = fund['date'].iloc[-1]
    
    cagr_dict = {'amfi_code': code}
    
    for years in [1, 3, 5]:
        start_date = end_date - pd.DateOffset(years=years)
        start_data = fund[fund['date'] <= start_date]
        
        if len(start_data) > 0:
            start_nav = start_data['nav'].iloc[-1]
            cagr = (end_nav / start_nav) ** (1/years) - 1
            cagr_dict[f'cagr_{years}yr'] = cagr
        else:
            cagr_dict[f'cagr_{years}yr'] = None
    
    results.append(cagr_dict)

cagr_df = pd.DataFrame(results)
print(cagr_df.shape)
print(cagr_df.head())

(40, 4)
   amfi_code  cagr_1yr  cagr_3yr cagr_5yr
0     100016 -0.022243  0.012926     None
1     100025  0.037050  0.039164     None
2     100033  0.532324  0.324425     None
3     101206  0.479241  0.289677     None
4     101207 -0.239860 -0.041524     None


In [3]:
# Sharpe Ratio = (Portfolio Return - Risk-free Rate) / Std Dev × √252
Rf = 0.065  # 6.5% annual risk-free rate (RBI repo rate proxy)
Rf_daily = Rf / 252  # convert to daily

sharpe_results = []

for code in nav['amfi_code'].unique():
    fund_returns = nav[nav['amfi_code'] == code]['daily_return'].dropna()
    
    mean_return = fund_returns.mean()
    std_return = fund_returns.std()
    
    sharpe = (mean_return - Rf_daily) / std_return * np.sqrt(252)
    
    sharpe_results.append({'amfi_code': code, 'sharpe_ratio': sharpe})

sharpe_df = pd.DataFrame(sharpe_results)
sharpe_df['rank'] = sharpe_df['sharpe_ratio'].rank(ascending=False)
sharpe_df = sharpe_df.sort_values('sharpe_ratio', ascending=False)

print(sharpe_df.shape)
print(sharpe_df.head(10))

(40, 3)
    amfi_code  sharpe_ratio  rank
34     148567      1.068224   1.0
30     120843      0.965561   2.0
36     148569      0.919047   3.0
25     120505      0.883256   4.0
19     119551      0.860977   5.0
38     149323      0.832885   6.0
2      100033      0.808268   7.0
9      118632      0.758851   8.0
16     119094      0.730547   9.0
3      101206      0.717409  10.0


In [4]:
sortino_results = []

for code in nav['amfi_code'].unique():
    fund_returns = nav[nav['amfi_code'] == code]['daily_return'].dropna()
    
    mean_return = fund_returns.mean()
    
    # Only consider NEGATIVE returns for downside deviation
    downside_returns = fund_returns[fund_returns < 0]
    downside_std = downside_returns.std()
    
    sortino = (mean_return - Rf_daily) / downside_std * np.sqrt(252)
    
    sortino_results.append({'amfi_code': code, 'sortino_ratio': sortino})

sortino_df = pd.DataFrame(sortino_results)
sortino_df['rank'] = sortino_df['sortino_ratio'].rank(ascending=False)
sortino_df = sortino_df.sort_values('sortino_ratio', ascending=False)

print(sortino_df.shape)
print(sortino_df.head(10))

(40, 3)
    amfi_code  sortino_ratio  rank
34     148567       1.490739   1.0
30     120843       1.479503   2.0
36     148569       1.352815   3.0
19     119551       1.291483   4.0
25     120505       1.285843   5.0
38     149323       1.167793   6.0
2      100033       1.144216   7.0
9      118632       1.098880   8.0
21     119598       1.067256   9.0
24     120504       1.063964  10.0


In [5]:
from scipy import stats

# Load benchmark data
benchmark = pd.read_csv(processed_path + "10_benchmark_indices_cleaned.csv")
benchmark['date'] = pd.to_datetime(benchmark['date'])

# Filter for Nifty 100 (check exact name in your data)
print(benchmark['index_name'].unique())

<ArrowStringArray>
[        'NIFTY50',        'NIFTY100', 'NIFTY_MIDCAP150',    'BSE_SMALLCAP',
        'NIFTY500',   'CRISIL_LIQUID',     'CRISIL_GILT']
Length: 7, dtype: str


In [6]:
from scipy import stats

# Get Nifty 100 returns
nifty100 = benchmark[benchmark['index_name'] == 'NIFTY100'].copy()
nifty100 = nifty100.sort_values('date')
nifty100['benchmark_return'] = nifty100['close_value'].pct_change()
nifty100 = nifty100[['date', 'benchmark_return']].dropna()

alpha_beta_results = []

for code in nav['amfi_code'].unique():
    fund_data = nav[nav['amfi_code'] == code][['date', 'daily_return']].dropna()
    
    # Merge fund returns with benchmark returns on matching dates
    merged = pd.merge(fund_data, nifty100, on='date', how='inner')
    
    if len(merged) > 30:  # need enough data points for meaningful regression
        slope, intercept, r_value, p_value, std_err = stats.linregress(
            merged['benchmark_return'], merged['daily_return']
        )
        
        beta = slope
        alpha = intercept * 252  # annualize
        
        alpha_beta_results.append({
            'amfi_code': code,
            'alpha': alpha,
            'beta': beta,
            'r_squared': r_value**2
        })

alpha_beta_df = pd.DataFrame(alpha_beta_results)
print(alpha_beta_df.shape)
print(alpha_beta_df.head(10))

(40, 4)
   amfi_code     alpha      beta  r_squared
0     100016  0.037476 -0.058268   0.002665
1     100025  0.042818  0.001158   0.000015
2     100033  0.271954  0.005104   0.000012
3     101206  0.213998  0.021086   0.000348
4     101207  0.108971 -0.065289   0.001064
5     101208  0.060861  0.000267   0.000046
6     102885  0.170488 -0.019487   0.000383
7     102886  0.028969 -0.042125   0.000896
8     102887  0.162113  0.016683   0.000186
9     118632  0.218294 -0.008354   0.000058


In [7]:
drawdown_results = []

for code in nav['amfi_code'].unique():
    fund = nav[nav['amfi_code'] == code].sort_values('date').copy()
    
    # Running maximum (highest NAV seen so far)
    fund['running_max'] = fund['nav'].cummax()
    
    # Drawdown = how far below the running max we currently are
    fund['drawdown'] = (fund['nav'] / fund['running_max']) - 1
    
    max_dd = fund['drawdown'].min()
    max_dd_date = fund.loc[fund['drawdown'].idxmin(), 'date']
    
    drawdown_results.append({
        'amfi_code': code,
        'max_drawdown': max_dd,
        'max_dd_date': max_dd_date
    })

drawdown_df = pd.DataFrame(drawdown_results)
print(drawdown_df.shape)
print(drawdown_df.head(10))

(40, 3)
   amfi_code  max_drawdown max_dd_date
0     100016     -0.247344  2022-09-15
1     100025     -0.043083  2023-07-28
2     100033     -0.162172  2022-05-12
3     101206     -0.112916  2023-07-05
4     101207     -0.354469  2026-05-11
5     101208     -0.001622  2023-09-12
6     102885     -0.108599  2022-03-29
7     102886     -0.280011  2026-04-27
8     102887     -0.215398  2022-07-04
9     118632     -0.174141  2024-07-19


In [8]:
# Merge all the dataframes we've built so far
scorecard = cagr_df[['amfi_code', 'cagr_3yr']].copy()
scorecard = scorecard.merge(sharpe_df[['amfi_code', 'sharpe_ratio']], on='amfi_code')
scorecard = scorecard.merge(alpha_beta_df[['amfi_code', 'alpha']], on='amfi_code')
scorecard = scorecard.merge(drawdown_df[['amfi_code', 'max_drawdown']], on='amfi_code')

# Get expense ratio from performance file
perf = pd.read_csv(processed_path + "07_scheme_performance_cleaned.csv")
scorecard = scorecard.merge(perf[['amfi_code', 'expense_ratio_pct']], on='amfi_code', how='left')

# Calculate ranks for each metric (1 = best)
scorecard['rank_3yr_return'] = scorecard['cagr_3yr'].rank(ascending=False)
scorecard['rank_sharpe'] = scorecard['sharpe_ratio'].rank(ascending=False)
scorecard['rank_alpha'] = scorecard['alpha'].rank(ascending=False)
scorecard['rank_expense'] = scorecard['expense_ratio_pct'].rank(ascending=True)  # lower expense = better, so ascending
scorecard['rank_maxdd'] = scorecard['max_drawdown'].rank(ascending=False)  # less negative = better, so ascending=False

# Convert ranks to a 0-100 scale (rank 1 = 100 points, rank 40 = lowest points)
n = len(scorecard)
scorecard['score_3yr'] = (n - scorecard['rank_3yr_return'] + 1) / n * 100
scorecard['score_sharpe'] = (n - scorecard['rank_sharpe'] + 1) / n * 100
scorecard['score_alpha'] = (n - scorecard['rank_alpha'] + 1) / n * 100
scorecard['score_expense'] = (n - scorecard['rank_expense'] + 1) / n * 100
scorecard['score_maxdd'] = (n - scorecard['rank_maxdd'] + 1) / n * 100

# Final weighted composite score
scorecard['fund_score'] = (
    0.30 * scorecard['score_3yr'] +
    0.25 * scorecard['score_sharpe'] +
    0.20 * scorecard['score_alpha'] +
    0.15 * scorecard['score_expense'] +
    0.10 * scorecard['score_maxdd']
)

scorecard = scorecard.sort_values('fund_score', ascending=False)

print(scorecard.shape)
print(scorecard[['amfi_code', 'fund_score']].head(10))

# Save deliverable
scorecard.to_csv("C:/Users/user/mutual-fund-analytics/reports/fund_scorecard.csv", index=False)
print("fund_scorecard.csv saved!")

(40, 17)
    amfi_code  fund_score
34     148567     86.2500
25     120505     82.8750
30     120843     82.0000
2      100033     80.7500
24     120504     79.3750
16     119094     78.2500
19     119551     74.1875
36     148569     73.6875
21     119598     68.0000
3      101206     67.5625
fund_scorecard.csv saved!


In [9]:
import plotly.express as px
import numpy as np

In [10]:
# Get top 5 funds based on fund_score
top5_funds = scorecard['amfi_code'].head(5).tolist()

# Get Nifty 50 and Nifty 100 returns
nifty50 = benchmark[benchmark['index_name'] == 'NIFTY50'].copy()
nifty50 = nifty50.sort_values('date')
nifty50['nifty50_return'] = nifty50['close_value'].pct_change()

nifty100_full = benchmark[benchmark['index_name'] == 'NIFTY100'].copy()
nifty100_full = nifty100_full.sort_values('date')
nifty100_full['nifty100_return'] = nifty100_full['close_value'].pct_change()

# Build comparison chart - cumulative returns over last 3 years
end_date = nav['date'].max()
start_date = end_date - pd.DateOffset(years=3)

fig = px.line(title="Top 5 Funds vs Nifty 50 & Nifty 100 (3 Year Performance)")

tracking_errors = []

for code in top5_funds:
    fund_data = nav[(nav['amfi_code'] == code) & (nav['date'] >= start_date)].sort_values('date')
    fund_data['cumulative_return'] = (1 + fund_data['daily_return'].fillna(0)).cumprod() - 1
    
    fig.add_scatter(x=fund_data['date'], y=fund_data['cumulative_return'], name=f"Fund {code}")
    
    # Tracking error vs Nifty 100
    merged_te = pd.merge(fund_data[['date', 'daily_return']], 
                          nifty100_full[['date', 'nifty100_return']], on='date', how='inner')
    diff = merged_te['daily_return'] - merged_te['nifty100_return']
    tracking_error = diff.std() * np.sqrt(252)
    tracking_errors.append({'amfi_code': code, 'tracking_error': tracking_error})

# Add benchmarks to chart
nifty50_period = nifty50[nifty50['date'] >= start_date].copy()
nifty50_period['cumulative_return'] = (1 + nifty50_period['nifty50_return'].fillna(0)).cumprod() - 1
fig.add_scatter(x=nifty50_period['date'], y=nifty50_period['cumulative_return'], 
                name="Nifty 50", line=dict(dash='dash', color='black'))

nifty100_period = nifty100_full[nifty100_full['date'] >= start_date].copy()
nifty100_period['cumulative_return'] = (1 + nifty100_period['nifty100_return'].fillna(0)).cumprod() - 1
fig.add_scatter(x=nifty100_period['date'], y=nifty100_period['cumulative_return'], 
                name="Nifty 100", line=dict(dash='dot', color='gray'))

fig.update_layout(height=600, yaxis_title="Cumulative Return")
fig.show()
fig.write_image("C:/Users/user/mutual-fund-analytics/reports/benchmark_comparison_chart.png")

# Save tracking error
te_df = pd.DataFrame(tracking_errors)
print(te_df)

   amfi_code  tracking_error
0     148567        0.187867
1     120505        0.232515
2     120843        0.206410
3     100033        0.224838
4     120504        0.187312


In [11]:
alpha_beta_df.to_csv("C:/Users/user/mutual-fund-analytics/reports/alpha_beta.csv", index=False)
print("alpha_beta.csv saved!")

alpha_beta.csv saved!


In [15]:
import pandas as pd
import numpy as np

processed_path = "C:/Users/user/mutual-fund-analytics/data/processed/"

# Reload nav data fresh
nav = pd.read_csv(processed_path + "02_nav_history_cleaned.csv")
nav['date'] = pd.to_datetime(nav['date'])
nav = nav.sort_values(['amfi_code', 'date'])
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()

# Now calculate std dev
std_dev_results = []

for code in nav['amfi_code'].unique():
    fund_returns = nav[nav['amfi_code'] == code]['daily_return'].dropna()
    annual_std = fund_returns.std() * np.sqrt(252)
    std_dev_results.append({'amfi_code': code, 'std_dev_annual': annual_std})

std_dev_df = pd.DataFrame(std_dev_results)
print(std_dev_df.shape)
print(std_dev_df.head())

(40, 2)
   amfi_code  std_dev_annual
0     100016        0.123004
1     100025        0.033040
2     100033        0.160291
3     101206        0.123321
4     101207        0.218130


In [18]:
scorecard_full = pd.read_csv("C:/Users/user/mutual-fund-analytics/reports/fund_scorecard.csv")
print("Before:", scorecard_full.columns.tolist())

if 'std_dev_annual_x' in scorecard_full.columns:
    scorecard_full['std_dev_annual'] = scorecard_full['std_dev_annual_x']
    scorecard_full = scorecard_full.drop(columns=['std_dev_annual_x', 'std_dev_annual_y'])

print("After:", scorecard_full.columns.tolist())
scorecard_full.to_csv("C:/Users/user/mutual-fund-analytics/reports/fund_scorecard.csv", index=False)
print("Fixed and saved!")

Before: ['amfi_code', 'cagr_3yr', 'sharpe_ratio', 'alpha', 'max_drawdown', 'expense_ratio_pct', 'rank_3yr_return', 'rank_sharpe', 'rank_alpha', 'rank_expense', 'rank_maxdd', 'score_3yr', 'score_sharpe', 'score_alpha', 'score_expense', 'score_maxdd', 'fund_score', 'std_dev_annual_x', 'std_dev_annual_y', 'std_dev_annual']
After: ['amfi_code', 'cagr_3yr', 'sharpe_ratio', 'alpha', 'max_drawdown', 'expense_ratio_pct', 'rank_3yr_return', 'rank_sharpe', 'rank_alpha', 'rank_expense', 'rank_maxdd', 'score_3yr', 'score_sharpe', 'score_alpha', 'score_expense', 'score_maxdd', 'fund_score', 'std_dev_annual']
Fixed and saved!
